# Least Square Fitting - Residuals

**Author**: SH

This notebook shows an interactive demonstration of residuals when looking at a simple linear regression model. The aim is to give a visual intuition of residuals - and what effects changing the model hyperparameters has in terms of the total sum of residuals - in lieu of having a whiteboard to sketch on.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from ipywidgets import interact
import ipywidgets as widgets

# Making it work if on Google Colab
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ModuleNotFoundError:
    pass


# Random number generator for white noise
rng = np.random.default_rng()

First, let's mock up our data - can just be a simple linear relationship:

$$ y_i = m x_i + C + n_i$$

where $m = 1.2$ and $C = 4$. We will also add some Gaussian white noise, $n$, to make the data look more representative. 

In [2]:
# Create ideal relation
x = np.array([1, 2, 4, 5, 7, 9])
m = 1.2
C = 4

y_true = m*x + C

# Add GWN to the data
y_meas = y_true + rng.normal(loc=0, scale=.5, size=len(x))

In [3]:
%matplotlib widget
fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(x, y_meas, 'o')
ax.set_xlim(0, 10)
ax.set_ylim(0, 20)
ax.set_xlabel('X')
ax.set_ylabel('Y')

# Setting up line of best fit
x_fit = np.linspace(0, 10, 128)
grad = 0.75
inter = 6

# Calculate expected values
y_expected = grad*x + inter
residuals = y_meas - y_expected
residual_plots = []
for x_m, y_m, y_exp in zip(x, y_meas, y_expected):
    buff, = ax.plot([x_m, x_m], [y_m, y_exp], 'r')
    residual_plots.append(buff)

# Plot line of 'best' fit
line, = ax.plot(x_fit, grad*x_fit + inter)

r_squared = (residuals**2).sum()
ax.set_title(f'Sum of Residuals Squared = {r_squared:0.2f}')

# Create function for slider
def update(gradient: float, intercept: float) -> None:
    '''Update line of best fit

    Inputs:
        gradient: gradient of linear plot
        intercept: intercept with y-axis

    Returns:
        None
    '''
    # Calculate new expected values
    y_exp = gradient*x + intercept

    # Update residual plots
    for i, ax_res in enumerate(residual_plots):
        ax_res.set_ydata((y_meas[i], y_exp[i]))

    # Update line of best fit
    line.set_ydata(gradient*x_fit + intercept)

    # Calculate residuals and update title
    resids = ((y_meas - y_exp)**2).sum()
    ax.set_title(f'Sum of Residuals Squared = {resids:0.2f}')

    # Update
    fig.canvas.draw_idle()

_ = interact(update,
             gradient=widgets.FloatSlider(value=grad,
                                          min=-5,
                                          max=5,
                                          step=0.1),
             intercept=widgets.FloatSlider(value=inter,
                                           min=-5,
                                           max=20,
                                           step=0.25))

fig.tight_layout()

RuntimeError: 'widget' is not a recognised GUI loop or backend name